# Day 22 · Projections — The Chicago Projection

**Question.** Chicago's address grid is the projection every plate in this atlas uses and the map Chicagoans
carry in their heads: 800 numbers to the mile from State & Madison, "eight blocks a mile". Tested the way a
cartographer tests a projection, where does it keep scale, and where does it stretch?

**Approach.** The city's street centerlines carry, for every street on the grid, its grid coordinate
(63rd Street = 6300 S, Western Avenue = 2400 W). Where two such streets meet, the corner has a real position
and an address position: a control point. A smooth rubber sheet through ~21,000 control points maps true miles
to "address miles" (address number ÷ 800). Pushed through it, the city outline, its streets and a circle at
every mile crossing (Tissot's indicatrix) show the projection's distortion.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, settings |
| 2. Pull | Street Center Lines from the Chicago Data Portal (dated snapshot, committed) |
| 3. Validate | Segments, grid coordinates, classes |
| 4. Transform | Corners with two grid streets → control points |
| 5. Analyze | Rubber sheet; numbers per mile, stretch by stretch; indicatrices; streets and outline |
| 6. QA | The sheet's fit; the old atlas rule against the real one; the lean |
| 7. Export | `plate.json` |

Run top to bottom from this folder.

## 1. Setup

In [ ]:
from __future__ import annotations

import gzip
import json
import logging
from datetime import date
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import distance_transform_edt

from atlas import Ledger, write_plate
from atlas.grid import address_south, miles_from_origin
from atlas.socrata import fetch

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
log = logging.getLogger('day22')
pd.set_option('display.width', 140)

In [ ]:
# Run settings ---------------------------------------------------------------------------------
REFRESH = False  # True = pull a new centerline snapshot

DAY = 22
DATASET_ID = 'pr57-gg9e'   # Street Center Lines (tabular twin of the map view 6imu-meau)
DATASET_NAME = 'Street Center Lines'
DATASET_URL = 'https://data.cityofchicago.org/d/6imu-meau'
DATASET_UPDATED = '2021-06-11'  # rowsUpdatedAt on the portal, read 2026-11-22
COLUMNS = 'objectid,fnode_id,tnode_id,pre_dir,street_nam,street_typ,class,ewns_dir,ewns_coord,the_geom'
STREET_CLASSES = {'1', '2', '3', '4'}  # expressways, arterials, collectors, residential streets
LATTICE = 0.25                         # miles between rubber-sheet nodes
WIN_ALONG, WIN_ACROSS = 1.0, 0.25      # miles: half-window along a street's axis and across it
MIN_POINTS = 5                         # control points a node needs
TISSOT_R = 0.3                         # miles: indicatrix radius
SIMPLIFY = 0.004                       # miles (about 20 feet)
INSET = {'west': -2.6, 'east': 1.0, 'south': -4.1, 'north': 0.25}  # true miles: Madison to 31st
WITHIN = 0.05                          # a mile keeps scale if its numbers are within 5% of 800
# The mile streets, north to south: Howard to 135th, with Roosevelt, Cermak and 31st
LADDER = [(7600, 'Howard'), (6400, 'Devon'), (5600, 'Bryn Mawr'), (4800, 'Lawrence'), (4000, 'Irving Park'),
          (3200, 'Belmont'), (2400, 'Fullerton'), (1600, 'North'), (800, 'Chicago'), (0, 'Madison'),
          (-1200, 'Roosevelt'), (-2200, 'Cermak'), (-3100, '31st'), (-3900, 'Pershing'), (-4700, '47th'),
          (-5500, '55th'), (-6300, '63rd'), (-7100, '71st'), (-7900, '79th'), (-8700, '87th'), (-9500, '95th'),
          (-10300, '103rd'), (-11100, '111th'), (-11900, '119th'), (-12700, '127th'), (-13500, '135th')]

DAY_DIR = Path.cwd()
REPO = DAY_DIR.parents[1]
assert (DAY_DIR / 'day22.ipynb').exists(), f'Run this notebook from its own folder, not {DAY_DIR}'
RAW_DIR = DAY_DIR / 'data' / 'raw'
OUT_PATH = DAY_DIR / 'output' / 'plate.json'
BOUNDARY_RAW = sorted((REPO / 'reference' / 'raw').glob('city_boundary_*.geojson'))[-1]

## 2. Pull
Every segment with its two end nodes, its street, class, grid coordinate (`ewns_dir` + `ewns_coord`: the
street's own address line, e.g. 63rd Street = S 6300) and geometry. Saved as a dated, gzipped CSV with the
coordinates as JSON (six decimals).

In [ ]:
existing = sorted(RAW_DIR.glob('street_centerlines_*.csv.gz'))
if REFRESH or not existing:
    raw = fetch(DATASET_ID, select=COLUMNS)
    raw['coords'] = [json.dumps([[[round(x, 6), round(y, 6)] for x, y in line] for line in c], separators=(',', ':'))
                     if isinstance(c, list) else '' for c in raw['the_geom.coordinates']]
    raw = raw.drop(columns=[c for c in raw.columns if c.startswith('the_geom')])
    pulled = date.today()
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    path = RAW_DIR / f'street_centerlines_{pulled.isoformat()}.csv.gz'
    raw.to_csv(path, index=False, compression='gzip')
else:
    path = existing[-1]
    pulled = date.fromisoformat(path.name.split('_')[-1].removesuffix('.csv.gz'))
seg = pd.read_csv(path, dtype=str, keep_default_na=False)
print(path.name, f'{len(seg):,} segments', pulled)

## 3. Validate

In [ ]:
seg = seg[seg['coords'] != ''].copy()
seg['lines'] = [[l for l in json.loads(c) if len(l) > 1] for c in seg['coords']]
seg = seg[seg['lines'].map(len) > 0].copy()
seg['coord'] = pd.to_numeric(seg['ewns_coord'], errors='coerce')
seg['signed'] = np.where(seg['ewns_dir'].isin(['S', 'W']), -seg['coord'], seg['coord'])
grid_y = seg['ewns_dir'].isin(['N', 'S']) & (seg['coord'] > 0)   # east-west streets: a north-south coordinate
grid_x = seg['ewns_dir'].isin(['E', 'W']) & (seg['coord'] > 0)   # north-south streets: an east-west coordinate
print(f'{len(seg):,} segments with geometry; {grid_y.sum():,} on east-west grid streets, {grid_x.sum():,} on north-south')
assert grid_y.sum() > 20_000 and grid_x.sum() > 20_000
print(seg['class'].value_counts().sort_index().to_string())
seg[grid_y].groupby('signed').size().loc[[-6300, -3100, -2200, -1200, 7600]]

## 4. Transform
A corner node is a control point when exactly one east-west grid coordinate and one north-south grid
coordinate meet there (a street that changes number at the node is left out). Its true position is the
node's coordinates in miles from State & Madison (`atlas.grid`); its address position is (east-west number,
north-south number) ÷ 800, signed, with the south side's real numbers: 6300 S = −7.875 address miles.

In [ ]:
ends = []
for f, t, lines in zip(seg['fnode_id'], seg['tnode_id'], seg['lines']):
    a, b = lines[0][0], lines[-1][-1]
    ends += [(f, a[0], a[1]), (t, b[0], b[1])]
nodes = pd.DataFrame(ends, columns=['node', 'lon', 'lat']).groupby('node').median()

def coords_at(mask):
    inc = pd.concat([seg.loc[mask, ['fnode_id', 'signed']].rename(columns={'fnode_id': 'node'}),
                     seg.loc[mask, ['tnode_id', 'signed']].rename(columns={'tnode_id': 'node'})])
    g = inc.groupby('node')['signed'].agg(['min', 'max'])
    return g.loc[g['min'] == g['max'], 'min']

cp = pd.concat({'Y': coords_at(grid_y), 'X': coords_at(grid_x)}, axis=1, join='inner').join(nodes)
cp['e'], cp['n'] = miles_from_origin(cp['lat'], cp['lon'])
cp['ax'], cp['ay'] = cp['X'] / 800, cp['Y'] / 800
city = shapely.transform(gpd.read_file(BOUNDARY_RAW).union_all(),
                         lambda c: np.column_stack(miles_from_origin(c[:, 1], c[:, 0])))
shapely.prepare(city)
cp['inCity'] = shapely.contains_xy(city, cp['e'], cp['n'])
print(f'{len(cp):,} control points, {cp["inCity"].sum():,} in the city')
cp.head()

## 5. Analyze
### 5a. How many numbers each mile holds
Each mile street's true position is the median of its control points in the city. Between two consecutive
mile streets: the numbers they span, the true miles, numbers per mile.

In [ ]:
cpc = cp[cp['inCity']]
pos = cpc.groupby('Y')['n'].median()
pos.loc[0] = 0.0
ladder = []
for (a, na), (b, nb) in zip(LADDER[:-1], LADDER[1:]):
    miles = float(pos[a] - pos[b])
    ladder.append({'from': na, 'to': nb, 'a': a, 'b': b, 'numbers': a - b, 'miles': round(miles, 3),
                   'perMile': round((a - b) / miles), 'top': round(float(pos[a]), 3)})
lad = pd.DataFrame(ladder)
lad['stretch'] = (lad['perMile'] / 800 - 1) > WITHIN * 1.5
print(lad.to_string())
south3 = lad[lad['a'].isin([0, -1200, -2200])]
assert list(south3['numbers']) == [1200, 1000, 900]
first3 = {'numbers': int(south3['numbers'].sum()), 'miles': round(float(-pos[-3100]), 2)}
first3['addressMiles'] = first3['numbers'] / 800
first3['extra'] = round(first3['addressMiles'] - first3['miles'], 2)
others = lad[~lad['a'].isin([0, -1200, -2200])]
print(first3, '| other stretches per mile', others['perMile'].min(), '-', others['perMile'].max())

### 5b. The rubber sheet
On a lattice every quarter mile, the shift from true miles to address miles is the median shift of the
control points in a window: for the north-south shift a band 2 mi wide and ½ mi tall (east-west streets
carry it), for the east-west shift the same band turned. Empty nodes take their nearest filled neighbour;
between nodes the shift is bilinear.

In [ ]:
ge = np.arange(-17, 8 + 1e-9, LATTICE)
gn = np.arange(-20, 12 + 1e-9, LATTICE)
E, N = cp['e'].to_numpy(), cp['n'].to_numpy()
DX, DY = (cp['ax'] - cp['e']).to_numpy(), (cp['ay'] - cp['n']).to_numpy()

def sheet(d, half_e, half_n):
    out = np.full((len(ge), len(gn)), np.nan)
    for i, x in enumerate(ge):
        col = np.abs(E - x) <= half_e
        for j, y in enumerate(gn):
            m = col & (np.abs(N - y) <= half_n)
            if m.sum() >= MIN_POINTS:
                out[i, j] = np.median(d[m])
    idx = distance_transform_edt(np.isnan(out), return_distances=False, return_indices=True)
    return out[tuple(idx)]

sx = sheet(DX, WIN_ACROSS, WIN_ALONG)
sy = sheet(DY, WIN_ALONG, WIN_ACROSS)
fx = RegularGridInterpolator((ge, gn), sx, bounds_error=False, fill_value=None)
fy = RegularGridInterpolator((ge, gn), sy, bounds_error=False, fill_value=None)

def to_addr(e, n):
    """True miles -> address miles (numbers / 800)."""
    p = np.column_stack([np.asarray(e, float).ravel(), np.asarray(n, float).ravel()])
    return p[:, 0] + fx(p), p[:, 1] + fy(p)

ax_, ay_ = to_addr(cpc['e'], cpc['n'])
err = np.hypot(ax_ - cpc['ax'], ay_ - cpc['ay'])
print(f'fit at the city\'s control points: median {np.median(err):.3f} address miles, p90 {np.quantile(err, .9):.3f}')

### 5c. Indicatrices
A circle of 0.3 mi at every mile crossing in the city, pushed through the sheet. Its height and width in
the projection, ÷ 0.6, are the scale in each direction; × 800, the numbers a mile holds there.

In [ ]:
th = np.linspace(0, 2 * np.pi, 33)[:-1]
tis = []
h = 0.05
for i in range(-15, 7):
    for j in range(-19, 12):
        if not city.contains(shapely.Point(i, j)):
            continue
        px, py = to_addr(i + TISSOT_R * np.cos(th), j + TISSOT_R * np.sin(th))
        cx_, cy_ = to_addr([i], [j])
        nsx = (to_addr([i], [j + h])[1] - to_addr([i], [j - h])[1])[0] / (2 * h)
        ewx = (to_addr([i + h], [j])[0] - to_addr([i - h], [j])[0])[0] / (2 * h)
        tis.append({'e': i, 'n': j, 'ax': round(float(cx_[0]), 3), 'ay': round(float(cy_[0]), 3),
                    'ns': round(800 * nsx), 'ew': round(800 * ewx),
                    'ring': [[round(float(x), 3), round(float(y), 3)] for x, y in zip(px, py)]})
tis = pd.DataFrame(tis)
tis['keeps'] = ((tis['ns'] / 800 - 1).abs() <= WITHIN) & ((tis['ew'] / 800 - 1).abs() <= WITHIN)
print(f'{len(tis)} indicatrices; {tis["keeps"].sum()} keep scale within {WITHIN:.0%} both ways')
tis.groupby(tis['n'].clip(-4, 1))[['ns', 'ew']].median()

### 5d. Streets, the city outline and the inset
Grid streets and the rest (classes 1–4: expressways to residential), merged by street and simplified, in
address miles for the map. The city outline in both. For the inset, Madison to 31st in true miles, with
Roosevelt, Cermak and 31st singled out.

In [ ]:
def lines_of(df):
    return [shapely.LineString(np.column_stack(miles_from_origin(np.array(l)[:, 1], np.array(l)[:, 0])))
            for ls in df['lines'] for l in ls if len(l) > 1]

streets = seg[seg['class'].isin(STREET_CLASSES)]
merged = []
for _, grp in streets.groupby(['pre_dir', 'street_nam', 'street_typ']):
    m = shapely.line_merge(shapely.union_all(lines_of(grp)))
    merged += list(getattr(m, 'geoms', [m]))

def addr_line(g):
    c = np.asarray(g.coords)
    x, y = to_addr(c[:, 0], c[:, 1])
    return shapely.simplify(shapely.LineString(np.column_stack([x, y])), SIMPLIFY)

r3 = lambda c: [[round(x, 3), round(y, 3)] for x, y in c]
map_streets = [r3(addr_line(g).coords) for g in merged if g.length > 0.02]
print(f'{len(map_streets):,} street lines, {sum(map(len, map_streets)):,} points')

def rings(geom, f=lambda c: c):
    out = []
    for p in getattr(geom, 'geoms', [geom]):
        for r in [p.exterior, *p.interiors]:
            out.append(r3(f(np.asarray(shapely.simplify(r, 0.01).coords))))
    return out

boundary_true = rings(city)
boundary_addr = rings(city, lambda c: np.column_stack(to_addr(c[:, 0], c[:, 1])))

box = shapely.box(INSET['west'], INSET['south'], INSET['east'], INSET['north'])
inset_lines, inset_hi = [], {}
for g in merged:
    c = g.intersection(box)
    for part in getattr(c, 'geoms', [c]):
        if part.length > 0.01:
            inset_lines.append(r3(shapely.simplify(part, 0.002).coords))
for key in (-1200, -2200, -3100):
    gl = shapely.union_all(lines_of(seg[grid_y & (seg['signed'] == key)])).intersection(box)
    inset_hi[str(key)] = [r3(shapely.simplify(p, 0.002).coords) for p in getattr(gl, 'geoms', [gl]) if p.length > 0.01]
print(f'inset: {len(inset_lines)} lines; highlighted', {k: len(v) for k, v in inset_hi.items()})

## 6. QA

In [ ]:
# The atlas's old rule (800 a mile after Roosevelt) against the real one, at the city's east-west grid streets
def old_rule_miles(a):
    a = np.asarray(a, float)
    return np.where(a >= 0, a / 800, np.where(a >= -1200, a / 1200, -1 + (a + 1200) / 800))

def real_rule_miles(a):
    a = np.asarray(a, float)
    return np.where(a >= 0, a / 800, -np.interp(-a, [0, 1200, 2200, 3100, 1e5], [0, 1, 2, 3, 3 + (1e5 - 3100) / 800]))

old = np.abs(cpc['n'] - old_rule_miles(cpc['Y']))
new = np.abs(cpc['n'] - real_rule_miles(cpc['Y']))
south = cpc['Y'] < -1200
qa = {'oldMedianMi': round(float(old[south].median()), 3), 'newMedianMi': round(float(new[south].median()), 3)}
print('south of Roosevelt, distance from where the rule puts a street:', qa)
assert qa['newMedianMi'] < qa['oldMedianMi'] / 2
# the site's label function agrees with the ladder at every mile street south of 31st, within 1/8 mile
gap = {a: round(float(address_south(-pos[a]) + a)) for a, _ in LADDER if a < -3100}
print('label minus real number at the mile streets:', gap)
assert max(map(abs, gap.values())) < 100, gap

# The lean: north-south grid streets drift east going south (and east-west ones drop going west)
def lean(by, along, across):
    slopes = []
    for _, t in cpc.groupby(by):
        if len(t) >= 20 and np.ptp(t[along]) > 3:
            slopes.append(np.polyfit(t[along], t[across], 1)[0])
    return float(np.degrees(np.arctan(np.median(slopes)))), len(slopes)
lean_ns = lean('X', 'n', 'e')  # degrees: east per mile north (negative = drifts east going south)
lean_ew = lean('Y', 'e', 'n')
qa['leanNS'], qa['leanEW'] = round(-lean_ns[0], 2), round(lean_ew[0], 2)
print(f'north-south streets lean {-lean_ns[0]:.2f} deg ({lean_ns[1]} streets); east-west {lean_ew[0]:.2f} deg ({lean_ew[1]})')

# 63rd Street and Howard, the plate's two examples
ex = {k: round(float(pos[k]), 2) for k in (-6300, 7600, -13000)} if -13000 in pos.index else {k: round(float(pos[k]), 2) for k in (-6300, 7600)}
qa['fitMedian'] = round(float(np.median(err)), 3)
print(ex, qa)

## 7. Export

In [ ]:
pct = lambda v: f'{round(100 * v)}%'
s = {
    'segments': len(seg), 'controlPoints': int(cp['inCity'].sum()), 'tissot': len(tis), 'keeps': int(tis['keeps'].sum()),
    'first3': first3, 'n63': round(float(-pos[-6300]), 2), 'a63': 6300 / 800,
    'howard': round(float(pos[7600]), 2), 'otherMin': int(others['perMile'].min()), 'otherMax': int(others['perMile'].max()),
    'lean': round((qa['leanNS'] + qa['leanEW']) / 2, 1),
}
finding = (f'Chicago’s address grid is a map projection: 800 numbers to the mile. It keeps that scale almost '
           f'everywhere, except in the first three miles south of Madison, which hold {first3["numbers"]:,} numbers. '
           f'So every South Side address counts about {first3["extra"]:.1f} mile more than the trip downtown.')
findings = [
    f'North of Madison, and from east to west, a mile holds 800 numbers, give or take 5%. Howard Street, 7600 N, '
    f'is {s["howard"]:.1f} miles from Madison, just as its number says.',
    f'South of Madison, the first mile holds 1,200 numbers (to Roosevelt Road, 1200 S), the second 1,000 (to Cermak '
    f'Road, 2200 S) and the third 900 (to 31st Street). Only after 31st does a mile hold 800.',
    f'So the numbers count {first3["addressMiles"]:.1f} miles where there are {first3["miles"]:.1f}. 63rd Street is '
    f'6300 S: at 800 a mile that says {s["a63"]:.1f} miles from Madison. It is {s["n63"]:.1f}.',
    'Until this plate, this atlas labeled its grid lines south of Roosevelt as if every mile held 800 numbers: the '
    'line at 63rd Street read 6000 S. Every plate now carries the real numbers.',
]
ledger = Ledger(
    source=f'{DATASET_NAME}, Chicago Data Portal', dataset_id=DATASET_ID, url=DATASET_URL, pulled=pulled,
    units='True miles from State & Madison; address miles = address number ÷ 800',
    method=[f'{s["controlPoints"]:,} corners where two grid streets meet: true position vs address position',
            f'Rubber sheet: median shift in windows on a {LATTICE} mi lattice, bilinear between',
            f'Indicatrices: circles of {TISSOT_R} mi at every mile crossing, pushed through the sheet'],
    caveats=[f'Centerlines last updated {DATASET_UPDATED}',
             f'Grid streets lean about {s["lean"]}° from true north; the sheet carries the lean'],
)
payload = {
    'summary': s, 'qa': qa, 'ladder': lad.to_dict('records'),
    'tissot': tis[['e', 'n', 'ax', 'ay', 'ns', 'ew', 'keeps', 'ring']].to_dict('records'),
    'streets': map_streets, 'boundaryTrue': boundary_true, 'boundaryAddr': boundary_addr,
    'inset': {'box': INSET, 'lines': inset_lines, 'hi': inset_hi,
              'real': {str(k): round(float(pos[k]), 3) for k in (-1200, -2200, -3100)}},
    'findings': findings, 'datasetUpdated': DATASET_UPDATED,
}
write_plate(OUT_PATH, day=DAY, ledger=ledger, finding=finding, payload=payload)
print(finding); print(OUT_PATH, f'{OUT_PATH.stat().st_size / 1e6:.2f} MB')